In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config # Pass the quantization configuration
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
input = tokenizer('What is the capital of india', return_tensors='pt')

In [ ]:
input

{'input_ids': tensor([[   1, 1824,  349,  272, 5565,  302, 1176,  515]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
input_on_device = {k: v.to(model.device) for k, v in input.items()}
outputs = model.generate(**input_on_device)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


What is the capital of india?
New Delhi


In [ ]:
#making an pipeline
from transformers import pipeline
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer)

In [ ]:
from langchain_huggingface import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
llm.invoke('what is 2 + 2 ?')

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'what is 2 + 2 ?\nA: 4'

## Step 1: Pre-Processing of .txt file

In [ ]:
with open('Company Overview.txt', 'r') as r:
  txt = r.read()

In [ ]:
import string

punctuations = string.punctuation


In [ ]:
def remove_unnessary_punctuation(t):
  punctuation = ['#', '&', '(', ')', '*', '-', '/', '?']
  if t in punctuation:
    t = t.replace(t, '')
  return t

In [ ]:
new_txt = ''.join(list(map(remove_unnessary_punctuation,txt)))

In [ ]:
new_txt

' Topic 1: Company Overview\n\n\n\n 1.1 Company Introduction\n\nXYZ Company is a private organization established in 2018 with the goal of providing highquality products and services in the technology and business solutions sector. The company focuses on innovation, customer satisfaction, and longterm growth. XYZ Company operates across multiple regions and serves both individual customers and corporate clients.\n\nThe organization believes in maintaining transparency, professionalism, and ethical standards in all business activities. Employees at XYZ Company are expected to contribute positively to the company’s objectives while following company policies and guidelines.\n\nXYZ Company continuously adapts to changing market trends and invests in modern technologies to improve service quality and operational efficiency.\n\n\n\n 1.2 Vision and Mission\n\n Vision\n\nTo become a trusted and leading company in the industry by delivering reliable, innovative, and customerfocused solutions w

In [ ]:
new_txt = new_txt.replace('\n\n','\n')

## Step 2: Chunking

In [ ]:
from semantic_router.encoders import HuggingFaceEncoder
from semantic_chunkers import StatisticalChunker

In [ ]:
encoder = HuggingFaceEncoder()
chunker = StatisticalChunker(min_split_tokens=500, max_split_tokens=800, encoder=encoder)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def chunking(txt):
  chunk = chunker(docs = [txt])
  chunk = list(map(lambda x: x.content, chunk[0]))
  return chunk

In [ ]:
chunks = chunking(new_txt)

2026-02-20 08:17:15 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 800. Splitting to sentences before semantically merging.


  0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
chunks

['Topic 1: Company Overview 1.1 Company Introduction XYZ Company is a private organization established in 2018 with the goal of providing highquality products and services in the technology and business solutions sector. The company focuses on innovation, customer satisfaction, and longterm growth. XYZ Company operates across multiple regions and serves both individual customers and corporate clients. The organization believes in maintaining transparency, professionalism, and ethical standards in all business activities. Employees at XYZ Company are expected to contribute positively to the company’s objectives while following company policies and guidelines. XYZ Company continuously adapts to changing market trends and invests in modern technologies to improve service quality and operational efficiency. 1.2 Vision and Mission Vision To become a trusted and leading company in the industry by delivering reliable, innovative, and customerfocused solutions while maintaining ethical busines

## Step - 3 Uploading on PineCone

In [ ]:
from pinecone import Pinecone

In [ ]:
PINECONE_KEY= "pcsk_2cia8Q_B5hpXyLBZXwHvkXyxfS3NSRRxi8VA5gV17HSdjGrvKi8ZmfgGZJhFgvkLe7hBbG"


In [ ]:
cilent = Pinecone(api_key=PINECONE_KEY)

In [ ]:
index = cilent.Index(host='dynamic-chatbot-2t52grx.svc.aped-4627-b74a.pinecone.io')

In [ ]:
for i, txt in enumerate(chunks):
  index.upsert_records(records = [{'id': 'doc ' + str(i), 'text': txt}], namespace='company')